# 02_modeling.ipynb

Model Training, Evaluation, and Validation

This notebook covers supervised and unsupervised modeling for learner question classification and clustering. It uses the cleaned data, builds TF-IDF features, trains a classifier, evaluates its performance (including confidence scores), and explores clustering for topic discovery.

## Steps
1. Setup and imports
2. Load and clean data
3. Feature engineering (TF-IDF, clustering)
4. Train/test split
5. Model training
6. Model evaluation (accuracy, confusion matrix, confidence)
7. Clustering analysis
8. Save model and artifacts

> **Note:** This notebook is intended for reproducible, milestone-based modeling. All paths and imports are project-root relative.

In [ ]:
# Add project root to sys.path for module imports
import sys
from pathlib import Path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Project modules
from src.config import RAW_DIR, PROCESSED_DIR, RANDOM_SEED, MODEL_PATH
from src.data.load import load_questions, load_lessons, load_users
from src.data.clean import clean_all
from src.features.build_features import build_tfidf_features, build_clustering_features
from src.models.train import train_classifier, save_model
from src.models.evaluate import evaluate_model, get_confidence_scores
from src.utils import set_random_seed

set_random_seed(RANDOM_SEED)
sns.set(style="whitegrid")

## 1. Load and Clean Data

Ensure processed data exists. If not, run the cleaning step.

In [ ]:
import os

questions_path = Path(PROCESSED_DIR) / "learner_questions.csv"
lessons_path = Path(PROCESSED_DIR) / "lessons.csv"
users_path = Path(PROCESSED_DIR) / "users.csv"

# If processed files are missing, run cleaning
if not (questions_path.exists() and lessons_path.exists() and users_path.exists()):
    print("Processed data not found. Running clean_all()...")
    clean_all()

# Load cleaned data
df_questions = load_questions(processed=True)
df_lessons = load_lessons(processed=True)
df_users = load_users(processed=True)

print(f"Questions: {df_questions.shape}, Lessons: {df_lessons.shape}, Users: {df_users.shape}")

### Inspect cleaned questions data

In [ ]:
df_questions.head()

## 2. Prepare Labeled Data for Supervised Modeling

We will use only rows with a non-null `label` for supervised classification.

In [ ]:
# Filter for labeled data
df_labeled = df_questions[df_questions['label'].notnull()].copy()
print(f"Labeled questions: {df_labeled.shape[0]} / {df_questions.shape[0]}")
df_labeled['label'] = df_labeled['label'].astype(str).str.strip()
labels = sorted(df_labeled['label'].unique())
print(f"Labels: {labels}")

## 3. Feature Engineering: TF-IDF Vectorization

In [ ]:
# Build TF-IDF features for labeled questions
tfidf_matrix, tfidf_vectorizer = build_tfidf_features(df_labeled, text_col="text", random_state=RANDOM_SEED)
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

## 4. Train/Test Split

Split labeled data into train and test sets (stratified by label).

In [ ]:
from sklearn.model_selection import train_test_split

X = tfidf_matrix
y = df_labeled['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_SEED
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 5. Train Classifier

We use Logistic Regression as the default classifier.

In [ ]:
clf = train_classifier(X_train, y_train, model_type="logreg", random_state=RANDOM_SEED)
print(f"Trained classifier: {clf.__class__.__name__}")

## 6. Model Evaluation

Evaluate on the test set: accuracy, confusion matrix, per-class metrics, and confidence scores.

In [ ]:
results = evaluate_model(clf, X_test, y_test)
print(f"Test accuracy: {results['accuracy']:.3f}")

In [ ]:
# Confusion matrix plot
cm = results['confusion_matrix']
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Per-class metrics
pd.DataFrame(results['per_class']).T

In [ ]:
# Confidence scores on test set
confidences = get_confidence_scores(clf, X_test)
df_conf = pd.DataFrame({
    'true_label': y_test,
    'pred_label': clf.predict(X_test),
    'confidence': confidences
})
plt.figure(figsize=(7, 4))
sns.histplot(df_conf['confidence'], bins=10, kde=True)
plt.xlabel('Prediction Confidence')
plt.title('Distribution of Prediction Confidence (Test Set)')
plt.show()

### Low-confidence predictions

Inspect test samples where the model is least confident.

In [ ]:
low_conf = df_conf.nsmallest(5, 'confidence')
for idx, row in low_conf.iterrows():
    q_idx = df_labeled.index[X_test.indices[idx] if hasattr(X_test, 'indices') else idx]
    print(f"Q: {df_labeled.loc[q_idx, 'text']}")
    print(f"  True: {row['true_label']}, Pred: {row['pred_label']}, Confidence: {row['confidence']:.2f}\n")

## 7. Unsupervised Clustering (All Questions)

Cluster all questions (including unlabeled) to surface recurring topics.

In [ ]:
# Build TF-IDF for all questions
tfidf_all, tfidf_vec_all = build_tfidf_features(df_questions, text_col="text", random_state=RANDOM_SEED)

# Cluster (e.g., 6 clusters)
n_clusters = 6
cluster_labels, kmeans = build_clustering_features(df_questions, tfidf_all, n_clusters=n_clusters, random_state=RANDOM_SEED)
df_questions['cluster_id'] = cluster_labels
print(f"Cluster label counts:\n{df_questions['cluster_id'].value_counts().sort_index()}")

In [ ]:
# Show sample questions from each cluster
for cid in range(n_clusters):
    print(f"\nCluster {cid} (n={sum(df_questions['cluster_id']==cid)}):")
    sample = df_questions[df_questions['cluster_id']==cid].sample(n=min(3, sum(df_questions['cluster_id']==cid)), random_state=RANDOM_SEED)
    for _, row in sample.iterrows():
        print(f"  - {row['text']}")

### Visualize clusters (2D projection)

Use PCA for dimensionality reduction.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(tfidf_all.toarray())

plt.figure(figsize=(7, 5))
palette = sns.color_palette('tab10', n_clusters)
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=df_questions['cluster_id'], palette=palette, legend='full', alpha=0.7)
plt.title('Question Clusters (PCA projection)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 8. Save Model and Artifacts

Persist the trained classifier and vectorizer for use in the dashboard.

In [ ]:
save_model(clf, MODEL_PATH)
tfidf_vec_path = Path(MODEL_PATH).with_suffix('.tfidfvec')
joblib.dump(tfidf_vectorizer, tfidf_vec_path)
print(f"Saved classifier to {MODEL_PATH}\nSaved TF-IDF vectorizer to {tfidf_vec_path}")

## 9. Summary

- Cleaned and loaded data
- Built TF-IDF features
- Trained and evaluated a classifier (with confidence scores)
- Explored unsupervised clustering for topic discovery
- Saved model and vectorizer for dashboard use

Next steps: integrate these artifacts into the Streamlit dashboard and enable user feedback and intervention indicators.